# BMIN 5200 — Week 12 in-class exercise
## Auditing the model you built: three definitions of fair, and you can't have all three

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week12.ipynb)

**Time:** ~25 minutes · **Pairs with:** Bias & fairness in AI

### What you'll do
- Write the three fairness metrics from the deck yourself, in about twenty lines, with no fairness library
- Find the disparity that was planted in the Week 9 data generator, as a number
- Fix demographic parity by adjusting one group's threshold, then watch predictive parity break
- Delete the sensitive attribute, retrain, and see the disparity survive intact

### Why it matters
Every metric in this notebook is two integers divided by two other integers off a confusion matrix. The arithmetic is not where the difficulty lives. The difficulty is that "treat these two groups fairly" turns out to name at least three mutually incompatible requirements, and choosing between them is a clinical and ethical decision that no amount of modelling will make for you — which is exactly the position a health system is in when it decides who gets a transitional-care nurse.

Setup. There is deliberately no `%pip install fairlearn` here. Everything in this notebook is
arithmetic you can do in your head once you see it, and hiding it behind a library call would
teach you the wrong lesson about how hard fairness auditing is.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(5200)
pd.set_option("display.width", 130)

## The cohort and the model — both regenerated

Same synthetic cohort, same generator, same seed, same forest as Week 11 — 1,200 fictional
discharges, no real patient data. Read the printout carefully: partner-network
patients are readmitted at a rate close to everyone else's, but their charts show *less than half*
as many prior admissions. That is the flaw, stated in two numbers. Their illness is the same;
their documentation is not, because their prior admissions happened at hospitals whose records we
do not receive.

In [ ]:
def make_readmission_cohort(n_patients=1200, seed=5200):
    """Synthetic 30-day readmission cohort. Not real patient data.

    Identical in the Week 9, Week 11, and Week 12 notebooks — same code, same seed — so all
    three weeks work on exactly the same 1,200 discharges. Nothing is saved to disk between
    notebooks; each one regenerates the cohort inline. The function builds its own generator
    so the cohort does not depend on what else has drawn from `rng` above it.
    """
    rng = np.random.default_rng(seed)

    care_network = rng.choice(["in_network", "community_partner"],
                              size=n_patients, p=[0.68, 0.32])
    partner = care_network == "community_partner"

    age = np.clip(rng.normal(68, 12, n_patients), 40, 95).round(0)
    true_prior_admissions = rng.poisson(1.8, n_patients)
    hemoglobin = np.clip(rng.normal(12.4, 1.6, n_patients)
                         - 0.9 * (true_prior_admissions > 2), 7.0, 17.0)
    creatinine = np.clip(rng.lognormal(np.log(1.0), 0.32, n_patients), 0.4, 6.0)

    # Partner-network patients are likelier to be on Medicaid and to live farther out.
    ins_probs = np.where(partner[:, None],
                         np.array([0.18, 0.34, 0.48]),
                         np.array([0.52, 0.36, 0.12]))
    draw = rng.random(n_patients)
    insurance_type = np.array(["commercial", "medicare", "medicaid"])[
        (draw[:, None] > ins_probs.cumsum(axis=1)).sum(axis=1)]

    distance_from_hospital = np.round(
        np.clip(rng.gamma(2.0, np.where(partner, 9.0, 3.4)), 0.5, 90.0), 1)

    # Ground truth: risk is driven by the TRUE admission history, identically in both groups,
    # and never by distance from the hospital.
    log_odds = (-2.0
                + 0.62 * true_prior_admissions
                + 0.030 * (age - 68)
                - 0.26 * (hemoglobin - 12.4)
                + 0.55 * (creatinine - 1.0))
    risk = 1.0 / (1.0 + np.exp(-log_odds))
    readmitted_30d = (rng.random(n_patients) < risk).astype(int)

    # The planted flaw: only 45% of a partner-network patient's prior admissions reach our
    # chart, against 97% for patients who stay in network. The illness is the same; the
    # documentation is not.
    capture = np.where(partner, 0.45, 0.97)
    recorded_prior = rng.binomial(true_prior_admissions, capture)

    return pd.DataFrame({
        "age": age.astype(int),
        "prior_admissions": recorded_prior,
        "hemoglobin": hemoglobin.round(1),
        "creatinine": creatinine.round(2),
        "insurance_type": insurance_type,
        "distance_from_hospital": distance_from_hospital,
        "care_network": care_network,
        "readmitted_30d": readmitted_30d,
    })


FEATURES = ["age", "prior_admissions", "hemoglobin", "creatinine",
            "insurance_type", "distance_from_hospital", "care_network"]

cohort = make_readmission_cohort()
model_input = cohort[FEATURES].copy()
model_input["insurance_type"] = model_input["insurance_type"].map(
    {"commercial": 0, "medicare": 1, "medicaid": 2})
model_input["care_network"] = model_input["care_network"].map(
    {"in_network": 0, "community_partner": 1})
outcome = cohort["readmitted_30d"]

X_train, X_test, y_train, y_test = train_test_split(
    model_input, outcome, test_size=0.3, random_state=5200, stratify=outcome)

forest = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=1,
                                random_state=5200, n_jobs=-1)
forest.fit(X_train, y_train)
predicted_risk = forest.predict_proba(X_test)[:, 1]
group = cohort.loc[X_test.index, "care_network"].values

print(f"Held-out AUC {roc_auc_score(y_test, predicted_risk):.3f} — the Week 11 model, rebuilt.")
print(cohort.groupby("care_network").agg(
    patients=("readmitted_30d", "size"),
    readmission_rate=("readmitted_30d", "mean"),
    mean_recorded_prior_admissions=("prior_admissions", "mean")).round(3).to_string())

## Part 1 — Twenty lines of arithmetic

The hospital will assign a transitional-care nurse to any patient the model scores above 0.35 —
a threshold set by how many nurses exist, not by anything statistical. So a "prediction" here is
a real allocation of a scarce clinical resource, which is what makes the audit matter.

Four numbers per group, straight off a confusion matrix, each corresponding to a slide:
**selection rate** (demographic parity), **TPR** and **FPR** (equalized error rate / equalized
odds), and **PPV** (predictive parity — of the patients we flagged, how many really came back).

In [ ]:
THRESHOLD = 0.35
GROUPS = ["in_network", "community_partner"]


def audit(risk_scores, threshold_by_group):
    """Per-subgroup fairness metrics. Nothing here is more than a ratio of two counts."""
    rows = []
    for group_name in GROUPS:
        in_group = group == group_name
        truth = y_test.values[in_group]
        flagged = (risk_scores[in_group] >= threshold_by_group[group_name]).astype(int)

        true_positives = int(((flagged == 1) & (truth == 1)).sum())
        false_positives = int(((flagged == 1) & (truth == 0)).sum())
        false_negatives = int(((flagged == 0) & (truth == 1)).sum())
        true_negatives = int(((flagged == 0) & (truth == 0)).sum())

        rows.append({
            "group": group_name,
            "patients": int(in_group.sum()),
            "actually readmitted": round(truth.mean(), 3),
            "flagged": int(flagged.sum()),
            "selection_rate": round(flagged.mean(), 3),
            "TPR": round(true_positives / max(true_positives + false_negatives, 1), 3),
            "FPR": round(false_positives / max(false_positives + true_negatives, 1), 3),
            "PPV": round(true_positives / max(true_positives + false_positives, 1), 3),
        })
    return pd.DataFrame(rows).set_index("group")


same_threshold = {name: THRESHOLD for name in GROUPS}
baseline = audit(predicted_risk, same_threshold)
print(baseline.to_string())
for name in GROUPS:
    print(f"mean predicted risk, {name:18s}: {predicted_risk[group == name].mean():.3f}")

There is the harm, and it is not subtle. Partner-network patients are readmitted slightly *more*
often than in-network patients in this test set, and the model flags them at 0.289 against 0.464 —
they get a nurse about six times for every ten times a comparable in-network patient does. Their
true positive rate is 0.364 against 0.675: of the partner-network patients who really were
readmitted, the model caught barely a third.

Nothing in the model is looking at care network to do this. It is reading `prior_admissions`,
faithfully, from a chart that under-counts theirs.

## Part 2 — Three definitions, three different verdicts

The deck gives you three incompatible things "fair" can mean. **Demographic parity**: flag the
same proportion of each group. **Equalized odds**: equal TPR and equal FPR, so errors fall
equally on both groups. **Predictive parity**: equal PPV, so a flag means the same thing
whichever group the patient is in.

Fill in the two TODOs, then read the verdicts. The placeholders return 0.0, which would claim
perfect fairness on two of the three definitions — a useful reminder that a fairness metric
reported without its definition is worth nothing.

In [ ]:
def demographic_parity_gap(table):
    """Largest difference in selection rate between any two groups."""
    return table["selection_rate"].max() - table["selection_rate"].min()


def equalized_odds_gap(table):
    """The worse of the TPR gap and the FPR gap — errors should fall equally on both groups."""
    # TODO: return max(TPR gap, FPR gap), each computed like demographic_parity_gap above.
    return 0.0


def predictive_parity_gap(table):
    """Difference in PPV: does a flag mean the same thing in both groups?"""
    # TODO: return the PPV gap.
    return 0.0


def verdicts(table, label):
    print(f"{label}")
    print(f"  demographic parity gap : {demographic_parity_gap(table):.3f}"
          f"   (four-fifths ratio {table['selection_rate'].min() / table['selection_rate'].max():.2f})")
    print(f"  equalized odds gap     : {equalized_odds_gap(table):.3f}")
    print(f"  predictive parity gap  : {predictive_parity_gap(table):.3f}")


verdicts(baseline, "Baseline model, one threshold for everyone:")

With the TODOs filled in, the three definitions do not agree with each other about this model.
Demographic parity is badly violated (0.175, a four-fifths ratio of 0.62 — below the 0.8 rule of
thumb used in US employment law). Equalized odds is badly violated (0.311, driven by the TPR
gap). Predictive parity is nearly satisfied: PPV is 0.468 against 0.457, so a flag really does
mean about the same thing in both groups.

A vendor reporting only the third number would be telling the truth and describing a model that
denies half the eligible partner-network patients a nurse.

## Part 3 — Fix demographic parity, and predict before you run

We are about to enforce demographic parity the crudest way possible: keep the in-network
threshold at 0.35, and lower the partner-network threshold until the same *proportion* of each
group gets flagged. This is the "Consequence of Demographic Parity" slide, made of real numbers.

**Commit to answers before running the next cell, out loud, as a room:**
1. The partner-network TPR is currently 0.364. After the fix, does it go up, down, or stay put?
2. The partner-network PPV is currently 0.457. Which direction does it move, and why?
3. Which of the three fairness definitions will be *worse* after we have fixed demographic parity?

In [ ]:
# Flag the same share of each group: find the partner-network score cutoff that matches
# the in-network selection rate exactly.
target_rate = (predicted_risk[group == "in_network"] >= THRESHOLD).mean()
partner_scores = np.sort(predicted_risk[group == "community_partner"])[::-1]
n_to_flag = int(round(target_rate * len(partner_scores)))
partner_threshold = float(partner_scores[n_to_flag - 1])

adjusted_thresholds = {"in_network": THRESHOLD, "community_partner": partner_threshold}
adjusted = audit(predicted_risk, adjusted_thresholds)

print(f"in-network threshold        : {THRESHOLD:.3f}")
print(f"partner-network threshold   : {partner_threshold:.3f}  (lowered to match the rate)\n")
print(adjusted.to_string())
print()
verdicts(baseline, "BEFORE — one threshold:")
print()
verdicts(adjusted, "AFTER  — demographic parity enforced:")

Demographic parity gap went from 0.175 to about 0.001 — the fix worked. The TPR gap narrowed too,
from 0.311 to 0.175, so more of the partner-network patients who genuinely get readmitted now get
a nurse. That is a real improvement in a real harm.

And predictive parity, which the baseline model very nearly satisfied, is now broken: PPV moves
from 0.468 / 0.457 to 0.468 / 0.393. A flag now means something different depending on which
network the patient came through. We did not introduce this by being careless. Because the two
groups' *score distributions* differ, equalizing the selection rate necessarily pushes one group
deeper into its lower-scoring patients, and those patients are readmitted less often. You cannot
hold both fixed at once, and the deck's "No easy solutions" slide is naming exactly this.

In [ ]:
metrics_to_plot = ["selection_rate", "TPR", "FPR", "PPV"]
positions = np.arange(len(metrics_to_plot))

figure, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for axis, (table, title) in zip(axes, [(baseline, "One threshold for everyone"),
                                       (adjusted, "Demographic parity enforced")]):
    axis.bar(positions - 0.2, table.loc["in_network", metrics_to_plot].astype(float),
             width=0.4, label="in_network")
    axis.bar(positions + 0.2, table.loc["community_partner", metrics_to_plot].astype(float),
             width=0.4, label="community_partner")
    axis.set_xticks(positions)
    axis.set_xticklabels(metrics_to_plot, rotation=20)
    axis.set_title(title)
    axis.set_ylim(0, 0.8)
axes[0].set_ylabel("rate")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Part 4 — Blinding: delete the attribute and see what happens

The most common proposal in the room, every time, is to stop giving the model the sensitive
attribute. This is the "Blinding Methods" slide. Below we drop `care_network` entirely and
retrain the identical forest on the six remaining features, so the model has no way of knowing
which network a patient came through.

Note first that `care_network` was already the *least* important feature in last week's SHAP
ranking, at 0.008. Predict what dropping it does to the disparity before you look.

In [ ]:
blinded_features = [name for name in FEATURES if name != "care_network"]
blinded_forest = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=1,
                                        random_state=5200, n_jobs=-1)
blinded_forest.fit(X_train[blinded_features], y_train)
blinded_risk = blinded_forest.predict_proba(X_test[blinded_features])[:, 1]

blinded = audit(blinded_risk, same_threshold)
print(f"Blinded model AUC {roc_auc_score(y_test, blinded_risk):.3f}\n")
print(blinded.to_string())
print()
verdicts(baseline, "With care_network as a feature:")
print()
verdicts(blinded, "With care_network deleted:")

The disparity is unchanged. Selection rates go from 0.464 / 0.289 to 0.464 / 0.264 — if anything
slightly worse — and the TPR gap is untouched. Deleting the column removed our ability to *see*
the problem, not the problem.

Why: the sensitive attribute is still in the data, spread across the features we kept. Partner
patients live farther out and are four times as likely to be on Medicaid. Complete the TODO below
and measure how well those leftovers reconstruct the attribute we just deleted.

In [ ]:
partner_flag = (cohort["care_network"] == "community_partner").astype(int)

# TODO: replace ["age"] with all six blinded_features to see what the model can still recover.
proxy_columns = ["age"]

proxy_model = LogisticRegression(max_iter=2000)
proxy_model.fit(X_train[proxy_columns], partner_flag.loc[X_train.index])
proxy_auc = roc_auc_score(partner_flag.loc[X_test.index],
                          proxy_model.predict_proba(X_test[proxy_columns])[:, 1])

print(f"Recovering care_network from {proxy_columns}: AUC {proxy_auc:.3f}")
print("(0.5 would mean the deleted attribute is genuinely gone.)\n")
print(cohort.groupby("care_network").agg(
    mean_distance_miles=("distance_from_hospital", "mean"),
    share_on_medicaid=("insurance_type", lambda column: (column == "medicaid").mean()),
).round(3).to_string())

## Talk about it

1. You have to pick one definition and defend it to a hospital ethics committee. Transitional-care
   nursing is a scarce benefit, not a burden. Does that change which of demographic parity,
   equalized odds, and predictive parity you would enforce — and would your answer flip if the
   model were instead selecting patients for a costly and invasive workup?
2. The real defect is that partner-network charts are missing prior admissions. Fixing the metric
   never fixes that. What would it cost, in dollars and in years, to fix the data instead — and
   who at your institution would have to agree to pay for it?
3. Nothing in Week 11's explainability work would have found this. SHAP told us the model uses
   `prior_admissions`, which is correct and reassuring and completely beside the point. What does
   that tell you about the relationship between explainability and fairness as safeguards?

## What this closes

Week 9 planted a data-quality flaw in one line of a generator: `capture = np.where(partner, 0.45,
0.97)`. Week 11 explained the model in detail and the flaw never appeared, because the model's
reasoning was faithful to a chart that was wrong. Week 12 found it in the first table we printed,
using nothing but counts. Three weeks, one dataset, and the bug was in the data the whole time.

## Solutions

Completed versions of the TODOs, as markdown so they do not run.

**Part 2 — the two gap functions:**

```python
def equalized_odds_gap(table):
    tpr_gap = table["TPR"].max() - table["TPR"].min()
    fpr_gap = table["FPR"].max() - table["FPR"].min()
    return max(tpr_gap, fpr_gap)


def predictive_parity_gap(table):
    return table["PPV"].max() - table["PPV"].min()
```

Which gives, for the baseline model: demographic parity gap 0.175, equalized odds gap 0.311,
predictive parity gap 0.011. After enforcing demographic parity: 0.001, 0.175, and 0.075. One
metric fixed, one improved, one nearly seven times worse.

**Part 4 — the proxy columns:**

```python
proxy_columns = blinded_features
```

AUC climbs from 0.529 with age alone to 0.887 with all six. A logistic regression with no access
to `care_network` identifies partner-network patients almost as well as reading the column would.
Distance does most of the work — partner patients average 18.0 miles out against 6.3 — with
insurance type close behind, at 44.6% Medicaid against 11.8%.